## Curious ME

In [1]:
import subprocess, sys

# Pin the entire boto stack atomically — all four packages must be compatible
boto_stack = [
    "aiobotocore==3.7.0",
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
]

# Install boto stack with --no-deps to prevent pip from overriding the pins
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--force-reinstall"] + boto_stack)

# Install all other dependencies normally
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "aioitertools", "h5py", "geopandas", "xarray", "netCDF4", "pyarrow"])

print("All dependencies installed.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.45.39 requires botocore==1.43.39, but you have botocore 1.43.0 which is incompatible.
awscli 1.45.39 requires s3transfer<0.20.0,>=0.19.0, but you have s3transfer 0.17.1 which is incompatible.
sagemaker 2.257.3 requires attrs<26,>=24, but you have attrs 26.1.0 which is incompatible.


All dependencies installed.


In [2]:
import os
import time
import io
import re
import urllib.request

from concurrent.futures import ThreadPoolExecutor, as_completed

import statistics

import boto3
import s3fs
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

print("Imports successful.")

Imports successful.


In [3]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET             = "central-virginia-tree-canopy-project"
S3_PREFIX             = "GEDI/GEDI02_A/002/"     
GEDI_COUNTY_S3_PREFIX = "gedi-county-summary/"
GEDI_INFO_KEY = "data/gedi-folder/city-of-charlottesville/gedi02_A_forCityOfCharlottesville.txt"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"
OUTPUT_PARQUET    = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_multiyear-cville.parquet")
OUTPUT_NETCDF     = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_grid-cville.nc")
OUTPUT_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_summary-cville.csv")
OUTPUT_DETAILED_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_height-cville.csv")
OUTPUT_COUNTY_CANOPY_HEIGHT_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_canopy_height-cville.csv")

# ── Spatial bounds (City of Charlottesville jurisdiction study area) ───────────────────────
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = -78.5237, 38.0096, -78.4463, 38.0705

# ── SMAP grid resolution (~9 km) ─────────────────────────────────────────────
GRID_RES = 0.081

# ── Target jurisdictions (Name, State FIPS, County/Place FIPS, Type) ─────────
TARGET_JURISDICTIONS = [
    ("Albemarle",       "51", "003",   "county"),
    ("Augusta",         "51", "015",   "county"),
    ("Buckingham",      "51", "029",   "county"),
    ("Charlottesville", "51", "14968", "place"),
    ("Fluvanna",        "51", "065",   "county"),
    ("Greene",          "51", "079",   "county"),
    ("Louisa",          "51", "109",   "county"),
    ("Nelson",          "51", "125",   "county"),
    ("Rockingham",      "51", "165",   "county"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI source  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI_COUNTY_S3_PREFIX}")

Configuration loaded.
  GEDI source  : s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/
  Output dir   : /home/ec2-user/SageMaker/gedi_output
  S3 output    : s3://central-virginia-tree-canopy-project/gedi-county-summary/


In [4]:
def parse_year_from_filename(filename: str):
    """Extract the year from a standard GEDI filename (e.g., GEDI02_A_2022143...)."""
    year_match = re.search(r'GEDI02_A_(\d{4})', filename)
    if year_match:
        return int(year_match.group(1))
    return None


def fetch_boundary(name: str, state_fips: str, geo_id: str, geo_type: str) -> gpd.GeoDataFrame:
    """Fetch boundary GeoJSON directly from the US Census TIGERweb API."""
    if geo_type == "place":
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "Places_CouSub_ConCity_SubMCD/MapServer/4/query"
            f"?where=STATE='{state_fips}'+AND+PLACE='{geo_id}'"
            "&outFields=NAME,STATE,PLACE&f=geojson&outSR=4326"
        )
    else:
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "State_County/MapServer/11/query"
            f"?where=STATE='{state_fips}'+AND+COUNTY='{geo_id}'"
            "&outFields=NAME,STATE,COUNTY&f=geojson&outSR=4326"
        )
    print(f"  Fetching boundary for {name}...")
    with urllib.request.urlopen(url, timeout=20) as r:
        gdf = gpd.read_file(r)
    if gdf.empty:
        raise ValueError(f"No boundary found for {name} (FIPS {state_fips}/{geo_id})")
    gdf = gdf.set_crs("EPSG:4326")
    gdf['jurisdiction'] = name
    return gdf

def extract_h5_filename(url: str) -> str:
    """
    Extract just the .h5 file name (basename) from a full granule URL.
 
    Parameters
    ----------
    url : str
        Full URL to a .h5 granule, e.g.
        https://.../GEDI02_A_2025189013841_..._V002/GEDI02_A_2025189013841_..._V002.h5
 
    Returns
    -------
    str
        Just the filename, e.g. GEDI02_A_2025189013841_..._V002.h5
    """
    path = urlparse(url).path
    return posixpath.basename(path)
 
 
def read_url_list_from_s3(bucket: str, key: str, profile: str = None) -> list[str]:
    """
    Fetch a text object from S3 and return the .h5 filename from each non-empty line.
 
    Parameters
    ----------
    bucket : str
        S3 bucket name.
    key : str
        S3 object key (path within the bucket).
    profile : str, optional
        Named AWS CLI profile to use. If None, uses the default credential chain.
 
    Returns
    -------
    list[str]
        List of .h5 file names, one per non-blank line in the source file.
    """
    session = boto3.Session(profile_name=profile) if profile else boto3.Session()
    s3 = session.client("s3")
 
    try:
        #log.info("Fetching s3://%s/%s ...", bucket, key)
        print(f"Fetching s3://{bucket}/{key} ...")
        response = s3.get_object(Bucket=bucket, Key=key)
        body = response["Body"].read().decode("utf-8")
    except NoCredentialsError:
        #log.error("No AWS credentials found. Configure ~/.aws/credentials, "
        #           "set AWS_PROFILE, or pass --profile.")
        print("No AWS credentials found. Configure ~/.aws/credentials, set AWS_PROFILE, or pass --profile.")

        raise
    except ClientError as e:
        error_code = e.response.get("Error", {}).get("Code", "Unknown")
        if error_code == "NoSuchKey":
            #log.error("Object not found: s3://%s/%s", bucket, key)
            print(f"Object not found: s3://{bucket}/{key}")
        elif error_code in ("AccessDenied", "403"):
            print("TEST")
        else:
            #log.error("S3 error (%s) reading s3://%s/%s: %s", error_code, bucket, key, e)
            print(f"S3 error ({error_code}) reading s3://{bucket}/{key}: {e}")
        raise
 
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    filenames = [extract_h5_filename(line) for line in lines]
    #log.info("Read %d line(s) from s3://%s/%s and extracted %d .h5 filename(s)",
    #          len(lines), bucket, key, len(filenames))
    print(f"Read {len(lines)} line(s) from s3://{bucket}/{key} and extracted {len(filenames)} .h5 filename(s)")
    return filenames

# This update works

def process_one_file(key):
    """
    Download one GEDI .h5 granule from S3 into memory (single bulk transfer,
    not lazy range-requests -- HDF5's internal metadata reads make lazy
    access over a network filesystem far slower due to per-request latency),
    then apply spatial + quality filtering. Returns a list of per-beam
    DataFrames (possibly empty).
    """
    file_name = os.path.basename(key)
    results = []

    year = parse_year_from_filename(file_name)
    if not year:
        print(f"Warning: Could not parse year from {file_name}. Skipping.")
        return results

    try:
        s3_path = f"s3://{S3_BUCKET}/{key}"
        with fs.open(s3_path, "rb") as f:
            raw_bytes = f.read()

        with h5py.File(io.BytesIO(raw_bytes), "r") as hf:
            beams = [k for k in hf.keys() if k.startswith("BEAM")]
            for beam in beams:
                if "lat_lowestmode" not in hf[beam]:
                    continue

                lats = hf[f"{beam}/lat_lowestmode"][:]
                lons = hf[f"{beam}/lon_lowestmode"][:]
                spatial_mask = (
                    (lons >= MIN_LON) & (lons <= MAX_LON) &
                    (lats >= MIN_LAT) & (lats <= MAX_LAT)
                )
                if not np.any(spatial_mask):
                    continue

                quality     = hf[f"{beam}/quality_flag"][:][spatial_mask]
                sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
                rh98        = hf[f"{beam}/rh"][:, 98][spatial_mask]

                beam_df = pd.DataFrame({
                    "longitude":    lons[spatial_mask],
                    "latitude":     lats[spatial_mask],
                    "quality_flag": quality,
                    "sensitivity":  sensitivity,
                    "rh98":         rh98,
                    "year":         year,
                    "file_source":  file_name,
                    "beam":         beam
                })

                valid_df = beam_df[
                    (beam_df["quality_flag"] == 1) &
                    (beam_df["sensitivity"]  > 0.9) &
                    (beam_df["rh98"]         > 0)   &
                    (beam_df["rh98"]         < 100)
                ]
                if not valid_df.empty:
                    results.append(valid_df)

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

    return results

print("Helper functions defined.")

Helper functions defined.


In [5]:
from urllib.parse import urlparse
import posixpath

county_specific_gedi_files = []
filenames = read_url_list_from_s3(S3_BUCKET, GEDI_INFO_KEY, profile=None)

print(f"\n{len(filenames)} GEDI .h5 file name(s):\n")

for name in filenames:
    county_specific_gedi_files.append(name)
    print(name)

print(f"Found {len(county_specific_gedi_files)} GEDI County Specific HDF5 files.")
if county_specific_gedi_files:
    print(f"  First : {county_specific_gedi_files[0]}")
    print(f"  Last  : {county_specific_gedi_files[-1]}")


Fetching s3://central-virginia-tree-canopy-project/data/gedi-folder/city-of-charlottesville/gedi02_A_forCityOfCharlottesville.txt ...
Read 36 line(s) from s3://central-virginia-tree-canopy-project/data/gedi-folder/city-of-charlottesville/gedi02_A_forCityOfCharlottesville.txt and extracted 36 .h5 filename(s)

36 GEDI .h5 file name(s):

GEDI02_A_2025143022717_O36496_03_T07196_02_004_02_V002.h5
GEDI02_A_2025132001407_O36324_02_T00291_02_004_02_V002.h5
GEDI02_A_2025046163723_O35001_03_T09889_02_004_02_V002.h5
GEDI02_A_2025024013344_O34650_03_T07349_02_004_02_V002.h5
GEDI02_A_2024292160935_O33139_03_T03080_02_004_02_V002.h5
GEDI02_A_2024285122312_O33028_02_T05983_02_004_02_V002.h5
GEDI02_A_2024227112139_O32128_02_T10252_02_004_03_V002.h5
GEDI02_A_2024204031223_O31766_03_T01657_02_004_03_V002.h5
GEDI02_A_2024196232657_O31655_02_T08676_02_004_03_V002.h5
GEDI02_A_2023039141749_O23553_02_T02984_02_003_02_V002.h5
GEDI02_A_2023013004828_O23141_02_T10252_02_003_02_V002.h5
GEDI02_A_2022274174211_O2

In [6]:
print(f"Scanning s3://{S3_BUCKET}/{S3_PREFIX} for GEDI HDF5 files...Only use county specific files.")

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

h5_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".h5"):
            #Check for a match in the county_specific_gedi_files array
            target_filename = posixpath.basename(obj["Key"])
            if target_filename in county_specific_gedi_files:
                #Add target filename to h5_keys array
                h5_keys.append(obj["Key"])

print(f"Found {len(h5_keys)} GEDI HDF5 files.")
if h5_keys:
    print(f"  First : {h5_keys[0]}")
    print(f"  Last  : {h5_keys[-1]}")

Scanning s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/ for GEDI HDF5 files...Only use county specific files.
Found 36 GEDI HDF5 files.
  First : GEDI/GEDI02_A/002/GEDI02_A_2019178093048_O03052_02_T04560_02_003_01_V002.h5
  Last  : GEDI/GEDI02_A/002/GEDI02_A_2025143022717_O36496_03_T07196_02_004_02_V002.h5


In [7]:
# Per-file timing breakdown -- confirms whether variance is file-size driven
# (expected/benign) rather than a systemic slowdown (would need more digging).
# if file_timings:
#     times = [t for _, t, _ in file_timings]
#     sizes_mb = [b / (1024 * 1024) for _, _, b in file_timings]
#     print(f"\nPer-file time: min={min(times):.2f}s  max={max(times):.2f}s  "
#           f"median={statistics.median(times):.2f}s  mean={statistics.mean(times):.2f}s")
#     print(f"Per-file size: min={min(sizes_mb):.1f}MB  max={max(sizes_mb):.1f}MB  "
#           f"median={statistics.median(sizes_mb):.1f}MB")

#     slowest = sorted(file_timings, key=lambda x: x[1], reverse=True)[:5]
#     print("\nSlowest 5 files (name, seconds, MB):")
#     for name, t, b in slowest:
#         print(f"  {name}: {t:.2f}s, {b / (1024 * 1024):.1f}MB")


In [8]:
# # New Supporting Method to reduce time to completion...

# def process_one_file(key):
#     """
#     Open one GEDI .h5 granule directly from S3 (lazy/range-request reads,
#     no full-file download), apply spatial + quality filtering, and return
#     a list of per-beam DataFrames (possibly empty).
#     """
#     file_name = os.path.basename(key)
#     results = []

#     year = parse_year_from_filename(file_name)
#     if not year:
#         print(f"Warning: Could not parse year from {file_name}. Skipping.")
#         return results

#     try:
#         s3_path = f"s3://{S3_BUCKET}/{key}"
#         # Hand h5py the open file-like object directly instead of f.read()ing
#         # it into BytesIO first -- h5py will issue range requests and only
#         # pull the byte ranges/chunks it actually touches.
#         with fs.open(s3_path, "rb") as f:
#             with h5py.File(f, "r") as hf:
#                 beams = [k for k in hf.keys() if k.startswith("BEAM")]
#                 for beam in beams:
#                     if "lat_lowestmode" not in hf[beam]:
#                         continue

#                     # Step 1: Extract coordinates first for rapid spatial masking
#                     lats = hf[f"{beam}/lat_lowestmode"][:]
#                     lons = hf[f"{beam}/lon_lowestmode"][:]
#                     spatial_mask = (
#                         (lons >= MIN_LON) & (lons <= MAX_LON) &
#                         (lats >= MIN_LAT) & (lats <= MAX_LAT)
#                     )
#                     if not np.any(spatial_mask):
#                         continue

#                     # Step 2: Extract attributes only for points inside the bbox
#                     quality     = hf[f"{beam}/quality_flag"][:][spatial_mask]
#                     sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
#                     rh98        = hf[f"{beam}/rh"][:, 98][spatial_mask]

#                     beam_df = pd.DataFrame({
#                         "longitude":    lons[spatial_mask],
#                         "latitude":     lats[spatial_mask],
#                         "quality_flag": quality,
#                         "sensitivity":  sensitivity,
#                         "rh98":         rh98,
#                         "year":         year,
#                         "file_source":  file_name,
#                         "beam":         beam
#                     })

#                     # Step 3: Strict quality filtering
#                     valid_df = beam_df[
#                         (beam_df["quality_flag"] == 1) &
#                         (beam_df["sensitivity"]  > 0.9) &
#                         (beam_df["rh98"]         > 0)   &
#                         (beam_df["rh98"]         < 100)
#                     ]
#                     if not valid_df.empty:
#                         results.append(valid_df)

#     except Exception as e:
#         print(f"Error reading {file_name}: {e}")

#     return results


In [9]:
# fs = s3fs.S3FileSystem(anon=False)

# # This is a long running processing....let's time it
# all_dfs = []
# cell_start_time = time.time()
# completed = 0

# MAX_WORKERS = 8  # tune based on bandwidth / S3 throttling behavior

# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#     futures = {executor.submit(process_one_file, key): key for key in h5_keys}

#     for future in as_completed(futures):
#         key = futures[future]
#         try:
#             file_dfs = future.result()
#             all_dfs.extend(file_dfs)
#         except Exception as e:
#             print(f"Unhandled error processing {os.path.basename(key)}: {e}")

#         completed += 1
#         if completed % 10 == 0:
#             elapsed = time.time() - cell_start_time
#             print(f"Processed {completed}/{len(h5_keys)} files... "
#                   f"({elapsed:.2f}s elapsed total)")

# # Calculate and display total run time
# total_elapsed = time.time() - cell_start_time
# minutes, seconds = divmod(total_elapsed, 60)
# print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
# print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

In [8]:
fs = s3fs.S3FileSystem(anon=False)

all_dfs = []
cell_start_time = time.time()
completed = 0

MAX_WORKERS = 16  # network I/O bound; push higher if no throttling/errors appear

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_one_file, key): key for key in h5_keys}

    for future in as_completed(futures):
        key = futures[future]
        try:
            file_dfs = future.result()
            all_dfs.extend(file_dfs)
        except Exception as e:
            print(f"Unhandled error processing {os.path.basename(key)}: {e}")

        completed += 1
        if completed % 10 == 0:
            elapsed = time.time() - cell_start_time
            print(f"Processed {completed}/{len(h5_keys)} files... "
                  f"({elapsed:.2f}s elapsed total)")

total_elapsed = time.time() - cell_start_time
minutes, seconds = divmod(total_elapsed, 60)
print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

Processed 10/36 files... (128.22s elapsed total)
Processed 20/36 files... (220.76s elapsed total)
Processed 30/36 files... (315.42s elapsed total)

Extraction complete. Beams with valid data: 83
🚀 Total execution time: 6m 3.63s


In [ ]:
fs = s3fs.S3FileSystem(anon=False)

all_dfs = []
cell_start_time = time.time()
completed = 0

MAX_WORKERS = 24  # network I/O bound; push higher if no throttling/errors appear

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_one_file, key): key for key in h5_keys}

    for future in as_completed(futures):
        key = futures[future]
        try:
            file_dfs = future.result()
            all_dfs.extend(file_dfs)
        except Exception as e:
            print(f"Unhandled error processing {os.path.basename(key)}: {e}")

        completed += 1
        if completed % 10 == 0:
            elapsed = time.time() - cell_start_time
            print(f"Processed {completed}/{len(h5_keys)} files... "
                  f"({elapsed:.2f}s elapsed total)")

total_elapsed = time.time() - cell_start_time
minutes, seconds = divmod(total_elapsed, 60)
print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

Processed 10/36 files... (118.38s elapsed total)
Processed 20/36 files... (223.77s elapsed total)


In [10]:
# Timing by file

# def process_one_file(key):
#     """
#     Download one GEDI .h5 granule from S3 into memory (single bulk transfer,
#     not lazy range-requests -- HDF5's internal metadata reads make lazy
#     access over a network filesystem far slower due to per-request latency),
#     then apply spatial + quality filtering. Returns a list of per-beam
#     DataFrames (possibly empty).
#     """
#     file_name = os.path.basename(key)
#     results = []
#     t0 = time.time()
#     n_bytes = 0

#     year = parse_year_from_filename(file_name)
#     if not year:
#         print(f"Warning: Could not parse year from {file_name}. Skipping.")
#         return results, time.time() - t0, n_bytes

#     try:
#         s3_path = f"s3://{S3_BUCKET}/{key}"
#         with fs.open(s3_path, "rb") as f:
#             raw_bytes = f.read()
#             n_bytes = len(raw_bytes)

#         with h5py.File(io.BytesIO(raw_bytes), "r") as hf:
#             beams = [k for k in hf.keys() if k.startswith("BEAM")]
#             for beam in beams:
#                 if "lat_lowestmode" not in hf[beam]:
#                     continue

#                 lats = hf[f"{beam}/lat_lowestmode"][:]
#                 lons = hf[f"{beam}/lon_lowestmode"][:]
#                 spatial_mask = (
#                     (lons >= MIN_LON) & (lons <= MAX_LON) &
#                     (lats >= MIN_LAT) & (lats <= MAX_LAT)
#                 )
#                 if not np.any(spatial_mask):
#                     continue

#                 quality     = hf[f"{beam}/quality_flag"][:][spatial_mask]
#                 sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
#                 rh98        = hf[f"{beam}/rh"][:, 98][spatial_mask]

#                 beam_df = pd.DataFrame({
#                     "longitude":    lons[spatial_mask],
#                     "latitude":     lats[spatial_mask],
#                     "quality_flag": quality,
#                     "sensitivity":  sensitivity,
#                     "rh98":         rh98,
#                     "year":         year,
#                     "file_source":  file_name,
#                     "beam":         beam
#                 })

#                 valid_df = beam_df[
#                     (beam_df["quality_flag"] == 1) &
#                     (beam_df["sensitivity"]  > 0.9) &
#                     (beam_df["rh98"]         > 0)   &
#                     (beam_df["rh98"]         < 100)
#                 ]
#                 if not valid_df.empty:
#                     results.append(valid_df)

#     except Exception as e:
#         print(f"Error reading {file_name}: {e}")

#     return results, time.time() - t0, n_bytes

In [ ]:
# fs = s3fs.S3FileSystem(anon=False)

# all_dfs = []
# file_timings = []  # (file_name, elapsed_seconds, size_bytes)
# cell_start_time = time.time()
# completed = 0

# MAX_WORKERS = 24  # network I/O bound; push higher if no throttling/errors appear

# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#     futures = {executor.submit(process_one_file, key): key for key in h5_keys}

#     for future in as_completed(futures):
#         key = futures[future]
#         file_name = os.path.basename(key)
#         try:
#             file_dfs, elapsed, n_bytes = future.result()
#             all_dfs.extend(file_dfs)
#             file_timings.append((file_name, elapsed, n_bytes))
#         except Exception as e:
#             print(f"Unhandled error processing {os.path.basename(key)}: {e}")

#         completed += 1
#         if completed % 10 == 0:
#             elapsed = time.time() - cell_start_time
#             print(f"Processed {completed}/{len(h5_keys)} files... "
#                   f"({elapsed:.2f}s elapsed total)")

# total_elapsed = time.time() - cell_start_time
# minutes, seconds = divmod(total_elapsed, 60)
# print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
# print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

# # Per-file timing breakdown -- confirms whether variance is file-size driven
# # (expected/benign) rather than a systemic slowdown (would need more digging).
# if file_timings:
#     times = [t for _, t, _ in file_timings]
#     sizes_mb = [b / (1024 * 1024) for _, _, b in file_timings]
#     print(f"\nPer-file time: min={min(times):.2f}s  max={max(times):.2f}s  "
#           f"median={statistics.median(times):.2f}s  mean={statistics.mean(times):.2f}s")
#     print(f"Per-file size: min={min(sizes_mb):.1f}MB  max={max(sizes_mb):.1f}MB  "
#           f"median={statistics.median(sizes_mb):.1f}MB")

#     slowest = sorted(file_timings, key=lambda x: x[1], reverse=True)[:5]
#     print("\nSlowest 5 files (name, seconds, MB):")
#     for name, t, b in slowest:
#         print(f"  {name}: {t:.2f}s, {b / (1024 * 1024):.1f}MB")

In [7]:
# #  One more tweak to make

# def process_one_file(key):
#     """
#     Download one GEDI .h5 granule from S3 into memory (single bulk transfer),
#     apply spatial + quality filtering, and return per-beam DataFrames plus
#     per-file timing/size info for diagnosing any real (vs. noise) slowdowns.
#     """
#     file_name = os.path.basename(key)
#     results = []
#     t0 = time.time()
#     n_bytes = 0

#     year = parse_year_from_filename(file_name)
#     if not year:
#         print(f"Warning: Could not parse year from {file_name}. Skipping.")
#         return results, time.time() - t0, n_bytes

#     try:
#         s3_path = f"s3://{S3_BUCKET}/{key}"
#         with fs.open(s3_path, "rb") as f:
#             raw_bytes = f.read()
#         n_bytes = len(raw_bytes)

#         with h5py.File(io.BytesIO(raw_bytes), "r") as hf:
#             beams = [k for k in hf.keys() if k.startswith("BEAM")]
#             for beam in beams:
#                 if "lat_lowestmode" not in hf[beam]:
#                     continue

#                 lats = hf[f"{beam}/lat_lowestmode"][:]
#                 lons = hf[f"{beam}/lon_lowestmode"][:]
#                 spatial_mask = (
#                     (lons >= MIN_LON) & (lons <= MAX_LON) &
#                     (lats >= MIN_LAT) & (lats <= MAX_LAT)
#                 )
#                 if not np.any(spatial_mask):
#                     continue

#                 quality     = hf[f"{beam}/quality_flag"][:][spatial_mask]
#                 sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
#                 rh98        = hf[f"{beam}/rh"][:, 98][spatial_mask]

#                 beam_df = pd.DataFrame({
#                     "longitude":    lons[spatial_mask],
#                     "latitude":     lats[spatial_mask],
#                     "quality_flag": quality,
#                     "sensitivity":  sensitivity,
#                     "rh98":         rh98,
#                     "year":         year,
#                     "file_source":  file_name,
#                     "beam":         beam
#                 })

#                 valid_df = beam_df[
#                     (beam_df["quality_flag"] == 1) &
#                     (beam_df["sensitivity"]  > 0.9) &
#                     (beam_df["rh98"]         > 0)   &
#                     (beam_df["rh98"]         < 100)
#                 ]
#                 if not valid_df.empty:
#                     results.append(valid_df)

#     except Exception as e:
#         print(f"Error reading {file_name}: {e}")

#     return results, time.time() - t0, n_bytes

In [ ]:
# MAX_WORKERS = 16  # network I/O bound; push higher if no throttling/errors appear

# # Explicit connection pool sized to >= MAX_WORKERS so botocore reuses
# # connections instead of opening/discarding extras beyond its default pool of 10.
# fs = s3fs.S3FileSystem(
#     anon=False,
#     config_kwargs={
#         "max_pool_connections": max(32, MAX_WORKERS * 2),
#         "retries": {"max_attempts": 8, "mode": "adaptive"},
#     },
# )

# all_dfs = []
# file_timings = []  # (file_name, elapsed_seconds, size_bytes)
# cell_start_time = time.time()
# completed = 0

# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#     futures = {executor.submit(process_one_file, key): key for key in h5_keys}

#     for future in as_completed(futures):
#         key = futures[future]
#         file_name = os.path.basename(key)
#         try:
#             file_dfs, elapsed, n_bytes = future.result()
#             all_dfs.extend(file_dfs)
#             file_timings.append((file_name, elapsed, n_bytes))
#         except Exception as e:
#             print(f"Unhandled error processing {file_name}: {e}")

#         completed += 1
#         if completed % 10 == 0:
#             total_so_far = time.time() - cell_start_time
#             print(f"Processed {completed}/{len(h5_keys)} files... "
#                   f"({total_so_far:.2f}s elapsed total)")

# total_elapsed = time.time() - cell_start_time
# minutes, seconds = divmod(total_elapsed, 60)
# print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
# print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

# # Per-file timing breakdown -- confirms whether variance is file-size driven
# # (expected/benign) rather than a systemic slowdown (would need more digging).
# if file_timings:
#     times = [t for _, t, _ in file_timings]
#     sizes_mb = [b / (1024 * 1024) for _, _, b in file_timings]
#     print(f"\nPer-file time: min={min(times):.2f}s  max={max(times):.2f}s  "
#           f"median={statistics.median(times):.2f}s  mean={statistics.mean(times):.2f}s")
#     print(f"Per-file size: min={min(sizes_mb):.1f}MB  max={max(sizes_mb):.1f}MB  "
#           f"median={statistics.median(sizes_mb):.1f}MB")

#     slowest = sorted(file_timings, key=lambda x: x[1], reverse=True)[:5]
#     print("\nSlowest 5 files (name, seconds, MB):")
#     for name, t, b in slowest:
#         print(f"  {name}: {t:.2f}s, {b / (1024 * 1024):.1f}MB")

Processed 10/36 files... (156.23s elapsed total)
